# Week 1 · Day 4 — Lab 2
## Exploratory Analysis in Jupyter

> **AI Engineering Academy** · Gamut Technology Services

Jupyter is where you *understand* a dataset before building anything on it. Used
well, it's the fastest path from raw data to insight. Used carelessly, it's a
reproducibility minefield — the **hidden-state problem**, stale outputs, and
out-of-order execution silently corrupt results. This lab does both halves: the
disciplined **exploratory workflow** (inspect → validate → quality-check →
visualize) and the **habits** that keep a notebook trustworthy.

### ⚙️ Local API — no internet required
The setup cell starts the same local FastAPI server (`lab_api.py`) on
`http://127.0.0.1:8002`, fetches all 252 events, and loads them into a DataFrame
`df` for you. From here on, the work is pandas + matplotlib exploration.

### Learning objectives
1. Run the standard **inspection sequence**: `shape`, `dtypes`, `isnull().sum()`, `describe`, `sample`, `value_counts`, `duplicated`.
2. Use Jupyter **magics** (`%%time`, `%timeit`) to profile, and understand the kernel/namespace model.
3. Write a **`validate_schema`** guard that asserts expected columns and dtypes.
4. Produce a **data-quality report** and flag columns over a null threshold.
5. Create quick **visualizations** (histogram, bar, correlation heatmap, scatter).
6. Explain and defend against the **hidden-state problem** with "Restart & Run All".

### Time budget — ~88 min
| Segment | Time |
|---|---|
| Setup & fetch | 6 min |
| **A.** Inspection patterns | 14 min |
| **B.** Magics & the kernel model | 12 min |
| **C.** Schema validation | 14 min |
| **D.** Data-quality report | 14 min |
| **E.** Visualization | 14 min |
| **F.** Hidden state & reproducibility | 12 min |
| Wrap-up + stretch | 2 min |

### Files you need (beside this notebook)
- `lab_api.py` — the local practice API (started for you).
- `API_REFERENCE.md` — endpoint documentation.


In [ ]:
%pip install --upgrade matplotlib

In [ ]:
# --- Setup: start the local API, fetch events, build a DataFrame -----------
import os, time, requests
import numpy as np
import pandas as pd
from lab_api import start_server

os.environ.setdefault("LAB_API_KEY", "local-dev-key")
os.environ.setdefault("API_KEY", os.environ["LAB_API_KEY"])
BASE_URL = "http://127.0.0.1:8002"

server, _thread = start_server(port=8002)
for _ in range(50):
    try:
        if requests.get(f"{BASE_URL}/health", timeout=1).status_code == 200:
            break
    except requests.exceptions.RequestException:
        time.sleep(0.1)

def auth_headers():
    return {"Authorization": f"Bearer {os.environ['API_KEY']}", "Accept": "application/json"}

def fetch_all_events(per_page=100):
    """Provided for you (this is the Lab 1 pattern) — returns all event dicts."""
    records, page = [], 1
    with requests.Session() as s:
        s.headers.update(auth_headers())
        while True:
            r = s.get(f"{BASE_URL}/v1/events", params={"page": page, "per_page": per_page}, timeout=(5, 30))
            r.raise_for_status()
            batch = r.json()["data"]
            records.extend(batch)
            if len(batch) < per_page:
                break
            page += 1
    return records

records = fetch_all_events()
df = pd.DataFrame(records)
df["created_at"] = pd.to_datetime(df["created_at"])   # parse the ISO strings to datetime
print("Fetched", len(df), "rows into a DataFrame")

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

print("ready.")

## Part A — The inspection sequence  *(guided)*

Run these in the first cells of every exploratory notebook. They catch schema
surprises before you build on bad assumptions.


In [ ]:
print("shape:", df.shape)
print(df.dtypes)
print("\nnulls per column:")
print(df.isnull().sum())
df.sample(3, random_state=0)

### Exercise A1 — Profile the fetched data
Compute: `n_rows`, `n_cols` (from `df.shape`); `category_nulls` (nulls in the
`category` column); `dup_count` (number of duplicate rows via `df.duplicated().sum()`);
and `top_model` (the most common value in `model`).


💡 **Hint.** `df.shape` is `(rows, cols)`. `df["category"].isnull().sum()`.
`df.duplicated().sum()` counts rows that are exact copies of an earlier row.
`df["model"].value_counts().idxmax()` gives the most common model.


In [ ]:
n_rows, n_cols = None, None
category_nulls = None
dup_count = None
top_model = None

In [ ]:
check("A1: 252 rows, 11 columns", lambda: n_rows == 252 and n_cols == 11)
check("A1: category has 20 nulls", lambda: category_nulls == 20)
check("A1: found 2 duplicate rows", lambda: dup_count == 2)
check("A1: top model is atlas-pro", lambda: top_model == "atlas-pro")

## Part B — Magics & the kernel model

The kernel is a live Python process; every cell shares one namespace. Magics profile
and introspect that process. `%%time` times a whole cell; `%timeit` times one line
over many runs (great for comparing implementations).


In [ ]:
%%time
# time a whole cell — here, re-fetching from the API
_ = fetch_all_events()

In [ ]:
# %timeit compares implementations. Vectorized vs apply for the same result:
%timeit df["output_tokens"] * 2
%timeit df["output_tokens"].apply(lambda x: x * 2)

### Exercise B1 — Vectorized equals apply (but faster)
Compute a `chars_per_token` column two ways and confirm they're identical:
`cpt_vectorized` = `df["response_length"] / df["output_tokens"]` (vectorized), and
`cpt_apply` = the same via `df.apply(lambda r: r["response_length"] / r["output_tokens"], axis=1)`.
Set `same` to whether they're equal (use `.equals` after aligning names, or compare
numpy arrays).


💡 **Hint.** The vectorized form is a single division of two Series. Compare with
`np.allclose(cpt_vectorized.to_numpy(), cpt_apply.to_numpy())`. Both give the same
numbers; the vectorized version is dramatically faster (that's what `%timeit` shows).


In [ ]:
cpt_vectorized = None    # TODO: response_length / output_tokens (vectorized)
cpt_apply = None         # TODO: same via df.apply(..., axis=1)
same = None              # TODO: are they numerically identical?

In [ ]:
check("B1: vectorized and apply give the same numbers", lambda: same is True)
check("B1: result has one value per row", lambda: len(cpt_vectorized) == len(df))

## Part C — Schema validation

Before cleaning or analysis, assert the data is shaped the way you expect. A schema
guard catches upstream changes (a renamed column, an int that became a float when
nulls appeared) at the *source*, not three steps downstream.


### Exercise C1 — Write `validate_schema`
Implement `validate_schema(df, required)` where `required` maps column name →
expected dtype string. It should raise `AssertionError` if any required column is
missing or has the wrong dtype, and print `"Schema OK"` otherwise. Run it on `df`
with the `REQUIRED` dict below (it should pass), and store the pass result in
`schema_ok`.


💡 **Hint.** `missing = set(required) - set(df.columns); assert not missing`. Then
loop `for col, dtype in required.items(): assert str(df[col].dtype) == dtype`. Compare
against `str(df[col].dtype)`.


In [ ]:
REQUIRED = {
    "event_id": "int64",
    "model": "str",
    "category": "str",
    "score": "float64",
    "created_at": "datetime64[us]",
}

def validate_schema(df, required):
    ...  # TODO: assert no missing columns, then assert each dtype matches
    print("Schema OK")
    return True

schema_ok = None     # TODO: validate_schema(df, REQUIRED)

In [ ]:
check("C1: schema validation passes on df", lambda: schema_ok is True)

### Exercise C2 — Catch a broken schema
Prove the guard actually fires. Make `broken = df.copy()` and change `score` to a
string dtype (`broken["score"] = broken["score"].astype("str")`). Call
`validate_schema(broken, REQUIRED)` inside a `try/except AssertionError` and set
`caught` to `True` when it raises.


In [ ]:
broken = df.copy()
# TODO: corrupt the score dtype to str
caught = False
# TODO: call validate_schema in a try/except AssertionError and set caught = True

In [ ]:
check("C2: validate_schema raised on the wrong dtype", lambda: caught is True)

## Part D — Data-quality report

A per-column summary — dtype, null count, null %, distinct values, a sample value —
turns "looks fine" into evidence. Flag columns whose null rate crosses a threshold so
problems announce themselves.


### Exercise D1 — Build the report and flag high-null columns
Implement `data_quality_report(df)` returning a DataFrame indexed by column with
columns `dtype`, `null_count`, `null_pct` (percent, rounded to 2), and `unique_count`.
Build `report`, then set `high_null` to the subset of rows where `null_pct > 5`.
`category` should be the flagged column.


💡 **Hint.** Build it from `df.dtypes`, `df.isnull().sum()`,
`(df.isnull().mean()*100).round(2)`, and `df.nunique()`. Assemble with
`pd.DataFrame({...})`. Then `report[report["null_pct"] > 5]`.


In [ ]:
def data_quality_report(df):
    return pd.DataFrame({
        # TODO: "dtype", "null_count", "null_pct", "unique_count"
    })

report = None        # TODO: data_quality_report(df)
high_null = None     # TODO: rows of report with null_pct > 5

In [ ]:
check("D1: report has one row per column", lambda: len(report) == df.shape[1])
check("D1: report columns are correct",
      lambda: set(report.columns) == {"dtype", "null_count", "null_pct", "unique_count"})
check("D1: category is flagged as high-null", lambda: "category" in high_null.index)
check("D1: category null_pct is ~7.9", lambda: abs(report.loc["category", "null_pct"] - 7.94) < 0.5)

## Part E — Quick visualization

Plots reveal shape that tables hide: distributions, correlations, outliers. Keep them
fast and disposable during exploration.


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

df["score"].hist(bins=30, figsize=(7, 3))
plt.title("Score Distribution"); plt.xlabel("score"); plt.tight_layout()
plt.show()

### Exercise E1 — Correlation structure
Build `numeric_cols = ["score", "input_tokens", "output_tokens", "latency_ms",
"response_length"]`. Compute the correlation matrix `corr`. Then find the pair of
**different** columns with the highest correlation and store their names in
`top_pair` (a set of two names) and the value in `top_corr`. Also draw a scatter of
`output_tokens` vs `response_length` (they should be strongly related by
construction).


💡 **Hint.** `corr = df[numeric_cols].corr()`. To find the strongest off-diagonal
pair, `unstack()` the matrix, drop self-pairs (`corr==1` on the diagonal), and take
`idxmax()`. `response_length` is built from `output_tokens`, so expect that pair.


In [ ]:
numeric_cols = ["score", "input_tokens", "output_tokens", "latency_ms", "response_length"]
corr = None          # TODO: correlation matrix of numeric_cols
top_pair = None      # TODO: {colA, colB} with the highest off-diagonal correlation
top_corr = None      # TODO: that correlation value

# scatter (visual)
df.plot.scatter(x="output_tokens", y="response_length", alpha=0.3, figsize=(6, 4))
plt.title("output_tokens vs response_length"); plt.tight_layout(); plt.show()

In [ ]:
check("E1: corr is a 5x5 matrix", lambda: corr.shape == (5, 5))
check("E1: strongest pair is output_tokens & response_length",
      lambda: top_pair == {"output_tokens", "response_length"})
check("E1: that correlation is strong (> 0.7)", lambda: top_corr > 0.7)

## Part F — Hidden state & reproducibility

The most dangerous notebook bug: a variable computed from an **old** cell, still
sitting in the kernel after you changed the code that produced it. The notebook
*displays* the latest output of each cell, not what a clean top-to-bottom run would
produce. The only defense is **Restart & Run All**.


In [ ]:
# A staged re-creation of the trap, in one cell so it is reproducible:
raw = pd.Series([1, 2, 3, 4], name="raw")

scored = raw * 10            # first definition
mean_score = scored.mean()   # computed now -> 25.0
print("mean_score (from x10):", mean_score)

scored = raw * 100           # you later CHANGE the transform...
# ...but mean_score is NOT recomputed. It is now STALE.
print("mean_score is still:", mean_score, "  <- stale! (kernel holds the old value)")

### Exercise F1 — Recompute to break the staleness
Given the situation above (`scored` is now `raw * 100` but `mean_score` still holds
the old `25.0`), compute `fresh_mean` from the **current** `scored`. Set `is_stale`
to whether the old `mean_score` differs from `fresh_mean`. This is what "Restart &
Run All" guarantees automatically: every value reflects the final code, in order.


💡 **Hint.** `fresh_mean = scored.mean()` recomputes from the current definition
(`raw * 100` → mean 250.0). `is_stale = mean_score != fresh_mean`.


In [ ]:
fresh_mean = None    # TODO: recompute the mean from the CURRENT scored
is_stale = None      # TODO: does the old mean_score differ from fresh_mean?

In [ ]:
check("F1: fresh_mean reflects the current transform (250.0)", lambda: fresh_mean == 250.0)
check("F1: the old value was indeed stale", lambda: is_stale is True)

## Stretch goals *(for fast finishers)*

**S1 — De-duplicate.** Drop the exact duplicate rows into `deduped` with
`df.drop_duplicates()`. It should have 250 rows (252 − 2).

**S2 — Nested → tidy.** Fetch `/v1/articles/nested` (page 1, per_page 20), flatten
with `pd.json_normalize(data, sep="_")`, then `.explode("tags")` into `tags_long` so
each tag is its own row. Confirm it has more rows than the 20 source articles.


In [ ]:
# S1
deduped = None       # TODO: df.drop_duplicates()

# S2
r = requests.get(f"{BASE_URL}/v1/articles/nested", params={"page": 1, "per_page": 20},
                 headers=auth_headers(), timeout=10)
arts = pd.json_normalize(r.json()["data"], sep="_")
tags_long = None     # TODO: explode the tags column

In [ ]:
check("S1: de-dup leaves 250 rows", lambda: len(deduped) == 250)
check("S2: exploding tags increased the row count", lambda: len(tags_long) > 20)
check("S2: author fields were flattened", lambda: {"author_name", "author_id"} <= set(arts.columns))

## Wrap-up — what you can now do

- Run the inspection sequence and spot seeded surprises (nulls, duplicates) immediately.
- Profile with `%%time` / `%timeit` and reason about the shared-kernel namespace.
- Guard data with a `validate_schema` that asserts columns and dtypes.
- Produce a data-quality report and auto-flag high-null columns.
- Draw quick distributions, correlations, and scatters.
- Explain the hidden-state problem and defend against it with "Restart & Run All".

**Next:** Lab 3 — recognize when exploratory logic has stabilized, extract it into a
tested importable module, and hand clean data off as Parquet.


In [ ]:
server.should_exit = True
time.sleep(0.3)
print("Local API stopped.")